<a href="https://colab.research.google.com/github/gaben83/A-Geometric-Parametrization-of-Flavor-from-an-S1-Z2-Compact-Dimension/blob/main/Geo%20par%20fl%20fit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ezt a kódot használtam fithez import numpy as np
import numpy as np
from scipy.optimize import minimize

# =========================
# EXPERIMENTAL DATA
# =========================
M_EXP = np.array([2.16, 1270, 173000, 4.67, 93, 4180])
V_EXP = np.array([0.225, 0.041, 0.0037])

# =========================
# MODEL
# =========================
def get_yukawa_matrix(params, q_type='up'):
    a0, phi_val, sigma, w, theta_w, eps, du, dd, q, log_ku, log_kd = params

    # enforce positivity via log-param
    kappa_u = np.exp(log_ku)
    kappa_d = np.exp(log_kd)

    R = np.sqrt(2) * a0
    theta = np.array([0, 2*np.pi/3, 4*np.pi/3])

    if q_type == 'up':
        theta_L, theta_R = theta, theta + du
        kappa = kappa_u
    else:
        theta_L, theta_R = theta + eps, theta + eps + dd
        kappa = kappa_d

    Y = np.zeros((3, 3), dtype=complex)

    for i in range(3):
        for j in range(3):
            theta_avg = (theta_L[i] + theta_R[j]) / 2.0
            theta_diff = theta_L[i] - theta_R[j]

            W_f = np.exp(-w * (1 - np.cos(theta_L[i] - theta_w))) * \
                  np.exp(-w * (1 - np.cos(theta_R[j] - theta_w)))

            phi_t = a0 + R * np.cos(theta_avg + phi_val)

            overlap = np.exp(-(theta_diff**2) / (4 * sigma**2))
            phase = np.exp(1j * q * theta_diff)

            Y[i, j] = kappa * W_f * phi_t * overlap * phase

    return Y

# =========================
# OBJECTIVE
# =========================
def objective(params):
    a0, phi_val, sigma, w, theta_w, eps, du, dd, q, log_ku, log_kd = params

    # HARD physics constraints
    if a0 <= 0:
        return 1e10
    if sigma < 0.15 or sigma > 1.2:
        return 1e10
    if w < 0 or w > 8:
        return 1e10
    if not (-2*np.pi <= q <= 2*np.pi):
        return 1e10
    if not (0 <= eps <= 1 and 0 <= du <= 1 and 0 <= dd <= 1):
        return 1e10

    try:
        Yu = get_yukawa_matrix(params, 'up')
        Yd = get_yukawa_matrix(params, 'down')

        mu = np.sort(np.linalg.svd(Yu, compute_uv=False))
        md = np.sort(np.linalg.svd(Yd, compute_uv=False))
        m_mod = np.concatenate([mu, md])

        Uu, _, _ = np.linalg.svd(Yu)
        Ud, _, _ = np.linalg.svd(Yd)

        V_ckm = np.abs(np.conj(Uu.T) @ Ud)
        v_mod = np.array([V_ckm[0,1], V_ckm[1,2], V_ckm[0,2]])

        # chi2
        chi2_m = 5.0 * np.sum((np.log(m_mod / M_EXP))**2)
        chi2_v = np.sum(((v_mod - V_EXP) / np.array([0.01, 0.01, 0.003]))**2)

        # REGULARIZATION (critical)
        penalty = 0
        penalty += 0.01 * q**2
        penalty += 0.01 * np.exp(log_ku - 10)**2
        penalty += 0.01 * np.exp(log_kd - 7)**2

        return chi2_m + chi2_v + penalty

    except:
        return 1e10

# =========================
# RANDOM INITIALIZATION
# =========================
def random_initial():
    return np.array([
        np.random.uniform(5, 25),            # a0
        np.random.uniform(-np.pi, np.pi),    # phi
        np.random.uniform(0.2, 1.0),         # sigma
        np.random.uniform(0, 5),             # w
        np.random.uniform(0, 2*np.pi),       # theta_w
        np.random.uniform(0, 0.5),           # eps
        np.random.uniform(0, 0.5),           # du
        np.random.uniform(0, 0.5),           # dd
        np.random.uniform(-1, 1),            # q
        np.random.uniform(8, 11),            # log(kappa_u)
        np.random.uniform(5, 8)              # log(kappa_d)
    ])

# =========================
# MULTI-START + STORE BEST
# =========================
N_STARTS = 60
TOP_N = 5

solutions = []

for i in range(N_STARTS):
    guess = random_initial()

    res = minimize(
        objective,
        guess,
        method='L-BFGS-B',
        options={'maxiter': 2000}
    )

    print(f"Run {i+1:02d} | chi2 = {res.fun:.4f}")

    if res.fun < 20:  # only keep decent fits
        solutions.append(res)

# sort solutions
solutions = sorted(solutions, key=lambda x: x.fun)[:TOP_N]

print("\n" + "="*60)
print("TOP PHYSICAL SOLUTIONS")
print("="*60)

# =========================
# PRINT RESULTS
# =========================
def print_solution(res, idx):
    p = res.x
    a0, phi_val, sigma, w, theta_w, eps, du, dd, q, log_ku, log_kd = p
    ku, kd = np.exp(log_ku), np.exp(log_kd)

    Yu = get_yukawa_matrix(p, 'up')
    Yd = get_yukawa_matrix(p, 'down')

    mu = np.sort(np.linalg.svd(Yu, compute_uv=False))
    md = np.sort(np.linalg.svd(Yd, compute_uv=False))

    Uu, _, _ = np.linalg.svd(Yu)
    Ud, _, _ = np.linalg.svd(Yd)

    V = np.abs(np.conj(Uu.T) @ Ud)

    print(f"\n--- Solution #{idx} ---")
    print(f"chi2 = {res.fun:.6f}")

    print(f"a0={a0:.3f}, sigma={sigma:.3f}, w={w:.3f}, q={q:.3f}")
    print(f"kappa_u={ku:.1f}, kappa_d={kd:.1f}")

    print("Masses:")
    print(f"Up:    {mu}")
    print(f"Down: {md}")

    print("CKM:")
    print(f"|V_us|={V[0,1]:.4f}, |V_cb|={V[1,2]:.4f}, |V_ub|={V[0,2]:.4f}")

for i, sol in enumerate(solutions):
    print_solution(sol, i+1) #es ez lett az eredmeny TOP PHYSICAL SOLUTIONS
#============================================================
#--- Solution #1 ---
#chi2 = 5.298800
#a0=11.587, sigma=0.776, w=1.073, q=-0.758
#kappa_u=4971.1, kappa_d=219.4
#Masses:
#Up:   [2.15144304e+00 2.51070358e+03 8.67032488e+04]
#Down: [4.36203807e+00 8.32203314e+01 5.61142820e+03]
#CKM:
#|V_us|=0.2235, |V_cb|=0.0408, |V_ub|=0.0031
#--- Solution #2 ---
#chi2 = 5.632573
#a0=22.112, sigma=0.445, w=1.454, q=0.732
#kappa_u=11846.0, kappa_d=294.6
#Masses:
#Up:   [2.20856376e+00 1.77367201e+03 1.23510447e+05]
#Down: [2.36704686e+00 1.43430705e+02 4.99877877e+03]
#CKM:
#|V_us|=0.2238, |V_cb|=0.0497, |V_ub|=0.0020
#--- Solution #3 ---
#chi2 = 8.663839
#a0=21.200, sigma=0.538, w=3.410, q=0.903
#kappa_u=38190.9, kappa_d=18491.4
#Masses:
#Up:   [3.20893682e+00 1.39169586e+03 1.06621900e+05]
#Down: [3.99552733e+00 5.84948164e+01 4.38089205e+03]
#CKM:
#|V_us|=0.2239, |V_cb|=0.0308, |V_ub|=0.0074
#--- Solution #4 ---
#chi2 = 14.599002
#a0=211.669, sigma=0.240, w=1.739, q=-0.141
#kappa_u=34105.8, kappa_d=2526.8
#Masses:
#Up:   [2.17515373e+00 2.38464295e+03 8.69418095e+04]
#Down: [3.72217063e+00 5.91521765e+01 8.40391613e+03]
#CKM:
#|V_us|=0.2217, |V_cb|=0.0241, |V_ub|=0.0093
#--- Solution #5 ---
#chi2 = 16.641278
#a0=16.333, sigma=1.133, w=2.139, q=0.267
#kappa_u=10664.4, kappa_d=968.7
#Masses:
#Up:   [2.34645227e+00 2.17731753e+03 1.11429039e+05]
#Down: [1.91875529e+00 1.43608445e+02 1.37450350e+04]
#CKM:
#|V_us|=0.2315, |V_cb|=0.0464, |V_ub|=0.0001

Run 01 | chi2 = 5.8455
Run 02 | chi2 = 596.8730
Run 03 | chi2 = 44.4286
Run 04 | chi2 = 1227.8088
Run 05 | chi2 = 98.8113
Run 06 | chi2 = 358.0813
Run 07 | chi2 = 516.3944
Run 08 | chi2 = 10000000000.0000
Run 09 | chi2 = 10354.3925
Run 10 | chi2 = 54.1101
Run 11 | chi2 = 143.2582
Run 12 | chi2 = 697.6686
Run 13 | chi2 = 43.1534
Run 14 | chi2 = 8.9286
Run 15 | chi2 = 595.6465
Run 16 | chi2 = 68.8777
Run 17 | chi2 = 10000000000.0000
Run 18 | chi2 = 541.8587
Run 19 | chi2 = 1006.7281
Run 20 | chi2 = 10000000000.0000
Run 21 | chi2 = 107.7819
Run 22 | chi2 = 150.6345
Run 23 | chi2 = 10000000000.0000
Run 24 | chi2 = 2267.7852
Run 25 | chi2 = 537.8995
Run 26 | chi2 = 470.8691
Run 27 | chi2 = 399.6701
Run 28 | chi2 = 761.7891
Run 29 | chi2 = 553.8238
Run 30 | chi2 = 111.4869
Run 31 | chi2 = 416.5787
Run 32 | chi2 = 10000000000.0000
Run 33 | chi2 = 1025.3205
Run 34 | chi2 = 177.6349
Run 35 | chi2 = 555.1888
Run 36 | chi2 = 6597.4321
Run 37 | chi2 = 41.1834
Run 38 | chi2 = 70.2927
Run 39 | chi2 